# 01 · Search modes — one cognee memory, queried many ways

`cognee.add` + `cognee.cognify` built a knowledge graph + embeddings from Wikipedia in AgensGraph (the `cognee_wiki` database). Here we ask the **same question through six `SearchType`s** to see what cognee's memory layer gives beyond plain RAG.

> Run `build.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
from _common import config
config.require_openai_key(); config.quiet()          # quiet cognee's verbose logs
config.configure("cognee_wiki")
import cognee
from cognee.modules.search.types import SearchType
from cognee.infrastructure.databases.graph import get_graph_engine

def _name(node):
    if isinstance(node, dict):
        return str(node.get("name") or (node.get("text") or "")[:40] or node.get("id") or "?")
    return str(node)[:40]

def render(results):
    if isinstance(results, (str, bytes)) or not isinstance(results, (list, tuple)):
        results = [results] if results else []
    for r in results[:4]:
        if isinstance(r, (tuple, list)) and len(r) == 3:
            src, edge, tgt = r
            rel = edge.get("relationship_name") if isinstance(edge, dict) else str(edge)
            print(f"   ({_name(src)}) -[{rel}]-> ({_name(tgt)})")
        elif isinstance(r, dict):
            print("   " + str(r.get("text") or r.get("name") or r)[:150])
        else:
            print("   " + str(r).strip().replace(chr(10), " ")[:550])
m = await (await get_graph_engine()).get_graph_metrics(include_optional=False)
print("knowledge graph:", m["num_nodes"], "nodes,", m["num_edges"], "edges")

knowledge graph: 5664 nodes, 12895 edges


## Ask one question, many ways

All of these answer the **same question** — the contrast is the point. `GRAPH_COMPLETION` and its variants (`GRAPH_SUMMARY_COMPLETION`, `GRAPH_COMPLETION_COT`, `GRAPH_COMPLETION_CONTEXT_EXTENSION`) reason over the **graph**; `RAG_COMPLETION`/`CHUNKS` are the no-graph baseline; `INSIGHTS` returns raw triplets (no LLM); `SUMMARIES` returns the per-document summaries; `NATURAL_LANGUAGE` turns your question into Cypher.

In [2]:
question = "What is anarchism, and what ideas, people, and movements is it connected to?"
modes = ["GRAPH_COMPLETION", "GRAPH_SUMMARY_COMPLETION", "GRAPH_COMPLETION_COT",
         "GRAPH_COMPLETION_CONTEXT_EXTENSION", "RAG_COMPLETION", "INSIGHTS",
         "CHUNKS", "SUMMARIES", "NATURAL_LANGUAGE"]
for mode in modes:
    print(f"\n### {mode}")
    render(await config.search(query_text=question, query_type=getattr(SearchType, mode)))


### GRAPH_COMPLETION


   Anarchism is a political philosophy and movement that is skeptical of all forms of authority, aiming to abolish coercive and hierarchical institutions, including nation-states and capitalism, and replace them with stateless societies and voluntary associations. It is primarily associated with libertarian socialism and has historical connections to workers' struggles, particularly during the 19th and early 20th centuries. Key historical events include the Paris Commune and significant participation in revolutions such as the Russian and Spanish 

### GRAPH_SUMMARY_COMPLETION


   Anarchism is a political philosophy advocating for the abolition of coercive institutions and the establishment of stateless societies and voluntary associations, often linked to libertarian socialism and the far-left. It played a role in 19th and early 20th-century workers' struggles and is exemplified by the anarchist-influenced Paris Commune of 1871.   Anarcho-capitalism is a related but distinct anti-statist ideology that emphasizes private property, free markets, and self-ownership, backed by thinkers like Murray Rothbard and influenced by

### GRAPH_COMPLETION_COT


   Anarchism is a political philosophy and movement that is skeptical of all forms of authority and seeks to abolish coercive and hierarchical institutions, advocating for stateless societies and voluntary associations. It is traditionally associated with the far-left and linked to libertarian socialism.  Key ideas connected to anarchism include: - Advocacy for stateless societies and voluntary free associations. - Historical involvement in workers' struggles and significant influence in movements like the Paris Commune and Spanish Civil War.  Pro

### GRAPH_COMPLETION_CONTEXT_EXTENSION


   Anarchism is a political philosophy and movement that is skeptical of all forms of authority, aiming to abolish coercive institutions and promote stateless societies based on voluntary associations. It is typically aligned with the far-left and is often linked to libertarian socialism.  Key figures and ideas connected to anarchism include: - **Libertarian socialism**: A left-wing perspective on anarchism combining libertarian principles with socialist values. - **Workers' struggles**: Anarchists played a significant role in various workers' ema

### RAG_COMPLETION


...

   Anarchism is a political ideology that questions all forms of authority and seeks to eliminate coercive institutions, advocating for stateless societi
   Anarcho-capitalism advocates for a stateless society where private property, free markets, and self-ownership are central. It promotes voluntary excha
   Angst is a feeling of fear or anxiety, closely related to feelings of apprehension and insecurity. The term originated from Danish and Norwegian words
   Agrarianism is a philosophy that champions subsistence farming, family agriculture, and property ownership, emphasizing political decentralization. Su

### NATURAL_LANGUAGE



2026-06-23T21:24:41.599720 [error    ] Error executing query: {'message': "Error executing graph query: MATCH (a:EntityType {name: 'anarchism'})-->(b:Entity) RETURN b.name, b.description, b.updated_at LIMIT 10", 'detail': 'Cypher query must end with RETURN or update clause'} [NaturalLanguageRetriever]



2026-06-23T21:24:42.884730 [error    ] Error executing query: {'message': "Error executing graph query: MATCH (a:EntityType {name: 'anarchism'})-->(b:Entity) RETURN b.name, b.description, b.updated_at LIMIT 10", 'detail': 'Cypher query must end with RETURN or update clause'} [NaturalLanguageRetriever]



2026-06-23T21:24:44.082293 [error    ] Error executing query: {'message': "Error executing graph query: MATCH (a:EntityType {name: 'anarchism'})-->(b:Entity) RETURN b.name, b.description, b.updated_at LIMIT 10", 'detail': 'Cypher query must end with RETURN or update clause'} [NaturalLanguageRetriever]



2026-06-23T21:24:44.083248 [warning  ] Failed to get results after 3 attempts for query: 'What is anarchism, and what ideas, people, and mov...' [NaturalLanguageRetriever]


## `CYPHER` — query the graph directly

`SearchType.CYPHER` is different: you pass a **Cypher query** (not a question) and get rows straight from the AgensGraph-backed graph. cognee stores every node on the `"__Node__"` label.

In [3]:
cypher = 'MATCH (n:"__Node__") WHERE n.name IS NOT NULL RETURN n.name AS name LIMIT 5'
print(cypher)
for row in (await config.search(query_text=cypher, query_type=SearchType.CYPHER) or [])[:5]:
    print("  ", row)

MATCH (n:"__Node__") WHERE n.name IS NOT NULL RETURN n.name AS name LIMIT 5


   {'name': 'abduction (films)'}
   {'name': 'alien abduction'}
   {'name': 'abduction'}
   {'name': 'albedo'}
   {'name': 'master of arts'}


## How it was built

```python
await cognee.add(wiki_articles, dataset_name="wiki")
await cognee.cognify(["wiki"])   # LLM extracts entities + relationships, summarizes, embeds
```

Graph + vectors both live in one AgensGraph database. Re-run `build.py` to (re)build.